# Mol-JEPA Evaluation

In this notebook we perform cross-validation, downstream probing and save the results for further analysis.
Note that we use the Mol-JEPA Inference module, which is more convenient during back and forth training adjustments.
For general evaluations we recommend to use the huggingface model directly.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Load paths
CHECKPOINT_DIR = os.getenv("CHECKPOINT_DIR")
LOGS_DIR = os.getenv("LOGS_DIR")

Load model weights

In [ ]:
from inference import MolJEPAInference
from eval_utils import find_checkpoint_for_version

# Important: Checkpoint file needs to match the configuration from the yaml, otherwise it will raise errors.
checkpoint_file = "/last.ckpt"
cfg = "moljepa_large.yaml"

# Option 1: Track back the stable-pretraining checkpoint for the logged run name
ckpt = find_checkpoint_for_version(f"{LOGS_DIR}/moljepa_large")
cpath = ckpt + checkpoint_file

# Option 2: Directly specify the checkpoint path
# cpath = f"{CHECKPOINT_DIR}final_jepa2/checkpoints/epoch=142-step=1007864.ckpt"

# Load checkpoint
model = MolJEPAInference(checkpoint_path=cpath, cfg=cfg)
predictions, cls, embeddings, attention = model(["CCCC", "CC=C"], return_attn=True)
print("Predictions shape: ", predictions.shape)
print("Embeddings shape: ", embeddings.shape)
print("CLS shape: ", cls.shape)
print("Attention shape: ", attention[0].shape)
print(model)

## Evaluation

### Config

In [ ]:
import os
import numpy as np
import pandas as pd
from config import CLUSTER_SPLITS

from eval_utils import (
    load_data_splits,
    load_public_split_data,
    featurize_dataset,
    compute_ecfp,
    run_torch_cross_validation,
    run_finetune_cross_validation,
    run_clamp_cross_validation,
    run_tabicl_cross_validation,
    run_lora_cross_validation,
    run_rf_cross_validation
)


# Select splits to train on
cv_datasets = CLUSTER_SPLITS.keys()
public_datasets = [
    ("asap_potency", "pIC50 (MERS-CoV Mpro)"),
    ("asap_potency", "pIC50 (SARS-CoV-2 Mpro)"),
    ("asap_admet", "log_HLM"),
    ("asap_admet", "log_KSOL"),
    ("asap_admet", "LogD"),
    ("asap_admet", "log_MDR1-MDCKII"),
    ("asap_admet", "log_MLM"),
    ("expansionrx", "log_Caco-2 Permeability Efflux"),
    ("expansionrx", "log_Caco-2 Permeability Papp A"),
    ("expansionrx", "log_HLM CLint"),
    ("expansionrx", "log_KSOL"),
    ("expansionrx", "LogD"),
    ("expansionrx", "log_MBPB"),
    ("expansionrx", "log_MGMB"),
    ("expansionrx", "log_MLM CLint"),
    ("expansionrx", "log_MPPB"),
    ("pxr", "pEC50")
]

# Select models to train
MODELS = {
    "mol-jepa": run_torch_cross_validation,
    "finetune": run_finetune_cross_validation,
    # "clamp": run_clamp_cross_validation, # clamp baselines were computed later outside of the other baselines
    "tabicl": run_tabicl_cross_validation,
    "lora": run_lora_cross_validation,
    "rf": run_rf_cross_validation,
}

# Run configuration
run_name = cfg.replace(".yaml", "")
MODE = "cv"  # cv or public
OVERWRITE = False
SAVE = False
POSTFIX = "74"  # 

if MODE == "cv":
    datasets = cv_datasets
    split_cols = ["split1", "split2", "split3"]
    results_dir = "cv_results"
elif MODE == "public":
    datasets = public_datasets
    split_cols = ["split1"]
    results_dir = "public_results"

### Run eval

In [ ]:
for model_name, cv_func in MODELS.items():
    print(f"Running {model_name}...")

    for dataset in datasets:
        print(f"----------{dataset}------------")
        results = []
        if MODE == "public":
            dataset_name, endpoint = dataset
            dataset = dataset_name + "_" + endpoint
        else:
            dataset_name = dataset

        # Check if results file already exists and skip
        results_file = (
            f"{results_dir}/{dataset}_{model_name}_{run_name}_{POSTFIX}_results.csv"
        )

        if os.path.exists(results_file) and not OVERWRITE:
            print(f"!!!! Results file {results_file} already exists. Skipping... !!!!!")
            continue

        # Use ROC-AUC for classification datasets
        metric = "roc_auc" if dataset_name == "tox21_linz" else "mae"
        print("-" * 10 + f" {dataset} (metric={metric}) " + "-" * 10)

        # Get embeddings for data
        if MODE == "public":
            df = load_public_split_data(dataset_name, endpoint)
        else:
            df = load_data_splits(dataset)

        # Dropping nans
        print("Nan count in y: ", df["y"].isna().sum())
        df = df.dropna(subset=["y"])

        # Split statistics
        for split_col in split_cols:
            print(f"Train size ({split_col}): ", df[df[split_col] == "train"].shape[0])
            print(f"Test size ({split_col}): ", df[df[split_col] == "test"].shape[0])
        
        # Get embeddings and features
        X_embeddings, X_latent, X_cls = featurize_dataset(df, model)
        X_concat = np.concatenate([X_embeddings, X_latent], axis=1)
        X_ecfp = compute_ecfp(df["smiles"].values)
        y = df["y"].values

        # Check if split cols not nan else continue
        if df[split_cols].isna().any().any():
            print(f"WARNING: NaN values found in split columns for dataset {dataset}. Skipping this dataset.")
            continue

        # Run cross-validation for the selected model
        if model_name == "clamp":
            results_clamp = cv_func(df, y, split_cols=split_cols, metric=metric)
            results_clamp.update({"dataset": dataset, "features": "clamp"})
            results.append(results_clamp)
        elif model_name == "tabicl":
            results_tabicl = cv_func(df, X_cls, y, split_cols=split_cols, metric=metric)
            results_tabicl.update({"dataset": dataset, "features": "cls"})
            results.append(results_tabicl)
        elif model_name == "rf":
            results_rf, fi = cv_func(df, X_ecfp, y, split_cols=split_cols, metric=metric)
            results_rf.update({"dataset": dataset, "features": "ecfp"})
            results.append(results_rf)
        elif model_name == "mol-jepa":
            for variant in ["linear", "nonlinear", "transformer"]:
                for feature_type, X in [("embeddings", X_embeddings), 
                                        ("latent", X_latent), 
                                        ("cls", X_cls), 
                                        ("concat", X_concat)]:
                    print(f"Running Mol-JEPA ({variant}) on {feature_type} features...")
                    results_torch = cv_func(df, X, y, variant=variant, split_cols=split_cols, metric=metric)
                    results_torch.update({"dataset": dataset, "features": feature_type, "variant": variant})
                    results.append(results_torch)


        elif model_name == "finetune":
            results_finetune = cv_func(df, model, y, split_cols=split_cols, epochs=50)
            results_finetune.update({"dataset": dataset, "features": "finetuned"})
            results.append(results_finetune)
        elif model_name == "lora":
            results_lora = cv_func(df, model, y, split_cols=split_cols, epochs=100)
            results_lora.update({"dataset": dataset, "features": "lora"})
            results.append(results_lora)
        else:
            raise ValueError(f"Unknown model: {model_name}")
        
        # Save current results to CSV
        if SAVE:
            results_df = pd.DataFrame(results)
            results_df.to_csv(results_file, index=False)